<a href="https://colab.research.google.com/github/alsulami412-arch/Vibration-group/blob/main/Vibration_group.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Home page - Vibration Group
from IPython.display import HTML, display
from datetime import date

profile = {
"Name": "Mohammed Alsulami",
"Employee ID": "87216",
"Position": "Vibration Analyst",
"Company": "Saudi Energy",
}

cards = "".join(
f"<div style='background:#ffffff14;border:1px solid #ffffff26;border-radius:10px;padding:12px 16px;min-width:160px'>"
f"<div style='font-size:11px;letter-spacing:1px;text-transform:uppercase;color:#9fd8d2'>{k}</div>"
f"<div style='font-size:17px;font-weight:600;margin-top:4px'>{v}</div></div>"
for k, v in profile.items()
)

html = f"""<div style='font-family:Segoe UI,Arial,sans-serif;background:linear-gradient(135deg,#0b1f33,#12476b);color:#fff;border-radius:14px;padding:28px 32px'>
<div style='font-size:12px;letter-spacing:2px;color:#9fd8d2'>SAUDI ENERGY - CONDITION MONITORING</div>
<h1 style='margin:6px 0 4px;font-size:30px;color:#fff'>Vibration Group</h1>
<div style='color:#c9d6e2;font-size:14px'>Vibration analysis workspace - {date.today():%d %B %Y}</div>
<div style='display:flex;flex-wrap:wrap;gap:12px;margin-top:20px'>{cards}</div>
</div>"""

display(HTML(html))

## Vibration Reports – Before T&I
Data source: `Vibration Reports Before T&I.xlsx` in this repository.
Each area (**Unit 03**, **Unit 04**, **Common PH#2**) has its own full report below.

Condition colours are taken from the report (ISO 20816 assessment):
🟢 Green = Accepted · 🟡 Yellow = Alert · 🟠 Orange = Warning · 🔴 Red = Danger · ⚪ Not Evaluated

**How to use:** Runtime → Run all. Edit the settings in cell 2 (ISO limits, criticality) and re-run.

In [ ]:
# 1) Load the Excel report from GitHub and build one clean table
import os, re, urllib.request
import pandas as pd
import openpyxl

import urllib.parse

FILE = "Vibration Reports Before T&I.xlsx"
REPO = "alsulami412-arch/Vibration-group"   # private repo -> needs a token

def download_from_github():
    """Private repo: uses a GitHub token saved in Colab Secrets as GITHUB_TOKEN."""
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    url = f"https://api.github.com/repos/{REPO}/contents/{urllib.parse.quote(FILE)}?ref=main"
    headers = {"Accept": "application/vnd.github.raw", "User-Agent": "colab"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as r, open(FILE, "wb") as f:
            f.write(r.read())
        return True
    except Exception as e:
        print(f"GitHub download failed ({e}).",
              "" if token else "Add a token in Colab Secrets (🔑 left sidebar) named GITHUB_TOKEN.")
        return False

def upload_manually():
    try:
        from google.colab import files
    except ImportError:
        return False
    print(f"Please upload '{FILE}' (any .xlsx name is accepted):")
    up = files.upload()
    if not up:
        return False
    name = next(iter(up))
    if name != FILE:
        os.replace(name, FILE)
    return True

if not os.path.exists(FILE):
    if not (download_from_github() or upload_manually()):
        raise FileNotFoundError(f"'{FILE}' not found. Add GITHUB_TOKEN to Colab Secrets or upload the file.")
# NOTE: no sample/fallback data is ever generated – the analysis only runs on the real report.

AREA_NAMES = {
    "Vibration Report U#04": "Unit 04",
    "Vibration Report U#03": "Unit 03",
    "Vibration Report Common PH#2": "Common PH#2",
}
RGB_TO_CONDITION = {"FF00B050": "Green", "FFFFFF00": "Yellow", "FFFF0000": "Red"}
THEME_TO_CONDITION = {9: "Orange", 0: "Not Evaluated"}

def cell_condition(cell):
    fc = cell.fill.fgColor
    if fc.type == "rgb":
        return RGB_TO_CONDITION.get(fc.rgb, "Not Evaluated")
    if fc.type == "theme":
        return THEME_TO_CONDITION.get(fc.theme, "Not Evaluated")
    return "Not Evaluated"

READING = re.compile(r"(Motor|Pump|Fan)\s+(NDE|DE)\s+([HVA])\s*=\s*([\d.]+)\s*mm/s", re.I)

def parse_readings(text):
    """'Motor NDE H= 2.4 mm/s, Motor DE H= 1.6 mm/s' -> list of (point, value)"""
    out = []
    for comp, end, direction, val in READING.findall(str(text)):
        out.append((f"{comp.title()} {end.upper()} {direction.upper()}", float(val)))
    return out

wb = openpyxl.load_workbook(FILE)
rows = []
for ws in wb.worksheets:
    area = AREA_NAMES.get(ws.title, ws.title)
    # find the header row ("No.")
    header_row = next(r for r in range(1, ws.max_row + 1) if str(ws.cell(r, 1).value).strip() == "No.")
    for r in range(header_row + 1, ws.max_row + 1):
        no = ws.cell(r, 1).value
        if no is None or str(no).strip() == "":
            continue
        vib_text = ws.cell(r, 5).value
        readings = parse_readings(vib_text)
        rows.append({
            "Area": area,
            "No.": int(no),
            "Equipment": " ".join(str(ws.cell(r, 2).value).split()),
            "Functional Location": ws.cell(r, 3).value,
            "Finding": ws.cell(r, 4).value,
            "Vibration": vib_text,
            "Recommendation": ws.cell(r, 6).value,
            "Condition": cell_condition(ws.cell(r, 4)),
            "Max (mm/s)": max((v for _, v in readings), default=None),
            "Max point": max(readings, key=lambda x: x[1])[0] if readings else None,
        })

df = pd.DataFrame(rows)
CONDITION_ORDER = ["Green", "Yellow", "Orange", "Red", "Not Evaluated"]
df["Condition"] = pd.Categorical(df["Condition"], CONDITION_ORDER, ordered=True)
print(f"Loaded {len(df)} equipment items from {df['Area'].nunique()} areas")
df.head()

### Settings (editable assumptions)
The ISO limits and the criticality ranking are **assumptions**. Adjust them to your machine classes and plant criticality register.

In [ ]:
# 2) Settings – edit to match your plant
# ISO 20816-3 velocity zone limits (mm/s RMS), Group 2 machines (15-300 kW).
# Rigid foundation: A/B=1.4, B/C=2.8, C/D=4.5 | Flexible foundation: 2.3 / 4.5 / 7.1
FOUNDATION = "rigid"     # "rigid" or "flexible"
ISO_LIMITS = {"rigid": (1.4, 2.8, 4.5), "flexible": (2.3, 4.5, 7.1)}[FOUNDATION]

# Equipment criticality (A = high, B = medium, C = low): first keyword match wins
CRITICALITY_RULES = [
    ("condensate extraction", "A"), ("circulating water", "A"), ("forced draft", "A"),
    ("preheater", "A"), ("jacking oil", "A"), ("emergency oil", "A"), ("seal oil", "A"),
    ("auxiliary oil", "A"), ("turning oil", "A"), ("fuel oil", "B"),
    ("gas recirculation", "B"), ("air extraction", "B"), ("heater drain", "B"),
    ("service seawater", "B"), ("closed circuit cooling", "B"), ("bearing & seal water", "B"),
    ("flame scanner", "B"), ("ignitor booster", "C"), ("screen wash", "C"),
    ("steam condenser exhauster", "C"),
]
CRIT_WEIGHT = {"A": 3, "B": 2, "C": 1}
SEVERITY = {"Green": 1, "Yellow": 2, "Orange": 3, "Red": 4, "Not Evaluated": 2}  # unknown = medium

FAULT_RULES = [   # keyword in Finding -> fault mode
    ("vane pass", "Vane pass / hydraulic"), ("van pass", "Vane pass / hydraulic"),
    ("flow turbulence", "Flow turbulence"), ("misalignment", "Misalignment"),
    ("alignment", "Misalignment"), ("coupling", "Coupling fault"),
    ("unbalance", "Unbalance"), ("looseness", "Looseness"),
    ("bearing", "Bearing fault"), ("screw mesh", "Gear / screw mesh"),
]

### Reliability model (applied to every area)

In [ ]:
# 3) Reliability model
import numpy as np

def iso_zone(v):
    if pd.isna(v):
        return "n/a"
    a, b, c = ISO_LIMITS
    return "A" if v <= a else "B" if v <= b else "C" if v <= c else "D"

# report colours that are consistent with each ISO zone
ZONE_OK_COLORS = {"A": {"Green"}, "B": {"Green", "Yellow"}, "C": {"Yellow", "Orange"}, "D": {"Orange", "Red"}}
ZONE_RANK = {"Green": 1, "Yellow": 2, "Orange": 3, "Red": 4}
ZONE_NUM = {"A": 1, "B": 2, "C": 3, "D": 4}

def criticality(name):
    n = name.lower().replace(",", " ")
    n = " ".join(n.split())
    for kw, c in CRITICALITY_RULES:
        if kw in n:
            return c
    return "B"

def fault_modes(finding):
    f = str(finding).lower()
    if f in ("accepted", "not evaluated", "nan"):
        return []
    modes = []
    for kw, mode in FAULT_RULES:
        if kw in f and mode not in modes:
            modes.append(mode)
    if not modes and "vibration" in f:
        modes = ["High vibration (root cause TBD)"]
    return modes

def system_key(row):
    # "Pump, Condensate Extraction #401" -> "Unit 04 | Pump Condensate Extraction"
    base = re.sub(r"[#\d.]+", " ", row["Equipment"])
    base = re.sub(r"[,]+", " ", base)
    base = re.sub(r"\b(Vertical|Mt)\b", " ", base, flags=re.I)
    return f"{row['Area']} | " + " ".join(base.split()).title()

rel = df.copy()
rel["ISO zone"] = rel["Max (mm/s)"].apply(iso_zone)
rel["Criticality"] = rel["Equipment"].apply(criticality)
rel["Fault modes"] = rel["Finding"].apply(fault_modes)
rel["System"] = rel.apply(system_key, axis=1)
rel["Severity"] = rel["Condition"].astype(str).map(SEVERITY)

# ISO check vs the report colour
def iso_check(r):
    if r["ISO zone"] == "n/a":
        return "Not measured"
    cond = str(r["Condition"])
    if cond in ZONE_OK_COLORS[r["ISO zone"]]:
        return "Consistent"
    # report colour lower than the zone allows -> overall level is worse than the report says
    if ZONE_RANK.get(cond, 0) < min(ZONE_RANK[c] for c in ZONE_OK_COLORS[r["ISO zone"]]):
        return "ISO worse than report"
    return "Report worse than ISO (spectrum-based)"
rel["ISO check"] = rel.apply(iso_check, axis=1)

# Redundancy: how many units in the same system are healthy (Green/Yellow)?
healthy = rel["Condition"].isin(["Green", "Yellow"])
sys_stats = rel.groupby("System").agg(Units=("No.", "count"),
                                      Healthy=("Condition", lambda s: s.isin(["Green", "Yellow"]).sum()),
                                      Degraded=("Condition", lambda s: s.isin(["Orange", "Red"]).sum()),
                                      Not_evaluated=("Condition", lambda s: (s == "Not Evaluated").sum()))
rel = rel.merge(sys_stats[["Units", "Healthy"]], left_on="System", right_index=True)
# penalty when a degraded machine has no healthy standby in its system
rel["No healthy standby"] = (rel["Condition"].isin(["Orange", "Red"]) & (rel["Units"] > 1)
                             & (rel["Healthy"] == 0))

# Risk score = severity x criticality weight (+2 if no healthy standby, +1 per extra fault mode)
rel["Risk score"] = (rel["Severity"] * rel["Criticality"].map(CRIT_WEIGHT)
                     + 2 * rel["No healthy standby"]
                     + rel["Fault modes"].apply(lambda m: max(len(m) - 1, 0)))

def priority(r):
    if r["Condition"] == "Not Evaluated":
        return "Survey" if r["Criticality"] != "C" else "P4"
    s = r["Risk score"]
    return "P1" if s >= 10 else "P2" if s >= 6 else "P3" if s >= 3 else "P4"
rel["Priority"] = rel.apply(priority, axis=1)
PRIORITY_TEXT = {
    "P1": "Immediate – fix during this T&I",
    "P2": "High – plan within T&I scope / next window",
    "P3": "Monitor – shorten route interval",
    "P4": "Routine monitoring",
    "Survey": "Collect baseline before return to service",
}
rel["Action window"] = rel["Priority"].map(PRIORITY_TEXT)
print(pd.crosstab(rel["Area"], rel["Priority"]).reindex(columns=["P1", "P2", "P3", "P4", "Survey"], fill_value=0))

### Area report engine
This cell defines the report. The cells after it run it once for each area.

In [ ]:
# 4) Area report: KPI card, charts, priority list, redundancy, survey list
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import HTML, display

COLORS = {"Green": "#00B050", "Yellow": "#FFD600", "Orange": "#F79646",
          "Red": "#E53935", "Not Evaluated": "#B0BEC5"}
PRI_COLORS = {"P1": "#E53935", "P2": "#F79646", "P3": "#FFD600", "P4": "#00B050", "Survey": "#B0BEC5"}
ZONE_SHADE = {"A": "#00B050", "B": "#9CCC65", "C": "#F79646", "D": "#E53935"}
HI = {"Green": 100, "Yellow": 70, "Orange": 40, "Red": 0}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

def kpi_card(title, kpis, footer=""):
    cards = "".join(
        f"<div style='background:#ffffff14;border:1px solid #ffffff26;border-radius:10px;padding:10px 14px;min-width:135px'>"
        f"<div style='font-size:10px;letter-spacing:1px;text-transform:uppercase;color:#9fd8d2'>{k}</div>"
        f"<div style='font-size:22px;font-weight:700;margin-top:3px'>{v}</div></div>" for k, v in kpis.items())
    display(HTML(f"<div style='font-family:Segoe UI,Arial,sans-serif;background:linear-gradient(135deg,#0b1f33,#12476b);"
                 f"color:#fff;border-radius:14px;padding:20px 24px;margin:6px 0'>"
                 f"<div style='font-size:12px;letter-spacing:2px;color:#9fd8d2'>VIBRATION REPORT – BEFORE T&I</div>"
                 f"<h2 style='margin:4px 0 12px;color:#fff'>{title}</h2>"
                 f"<div style='display:flex;flex-wrap:wrap;gap:10px'>{cards}</div>"
                 f"<div style='margin-top:12px;font-size:13px;color:#dfe8f0'>{footer}</div></div>"))

def styled(frame, color_col, palette):
    fmt = {c: (lambda v: "" if pd.isna(v) else f"{v:g}") for c in frame.columns if c == "Max (mm/s)"}
    return (frame.style.apply(lambda r: [f"background-color:{palette.get(str(r[color_col]), '#fff')}33"] * len(r), axis=1)
                 .format(fmt).set_properties(**{"text-align": "left"}))

def area_report(area):
    a = rel[rel["Area"] == area].copy()
    n, ev = len(a), a[a["Condition"] != "Not Evaluated"]
    health = ev["Condition"].astype(str).map(HI).mean() if len(ev) else float("nan")
    bad = a["Condition"].isin(["Orange", "Red"])
    sys_a = sys_stats[sys_stats.index.str.startswith(area + " |")]
    lost = sys_a[(sys_a["Units"] > 1) & (sys_a["Healthy"] == 0) & (sys_a["Degraded"] > 0)]
    fm_a = a.explode("Fault modes").dropna(subset=["Fault modes"])["Fault modes"].value_counts()
    worst = ev.loc[ev["Max (mm/s)"].idxmax()] if ev["Max (mm/s)"].notna().any() else None

    # ---- KPI card ----
    kpi_card(area, {
        "Equipment": n,
        "Evaluated": f"{len(ev)} ({len(ev) / n:.0%})",
        "Health index": f"{health:.0f}/100",
        "Green / Yellow": int(a["Condition"].isin(["Green", "Yellow"]).sum()),
        "Orange / Red": int(bad.sum()),
        "P1 items": int((a["Priority"] == "P1").sum()),
        "Lost redundancy": len(lost),
    }, footer=(f"<b>Highest reading:</b> {worst['Equipment']} – {worst['Max (mm/s)']:g} mm/s ({worst['Max point']})"
               f" &nbsp;|&nbsp; <b>Dominant fault:</b> {fm_a.index[0] if len(fm_a) else '-'}") if worst is not None else "")

    # ---- Chart 1: every machine vs ISO zones (the key vibration chart) ----
    m = ev.dropna(subset=["Max (mm/s)"]).sort_values("Max (mm/s)")
    h = max(3.5, 0.32 * len(m) + 1.2)
    fig, ax = plt.subplots(figsize=(13, h))
    lim_a, lim_b, lim_c = ISO_LIMITS
    xmax = max(m["Max (mm/s)"].max() * 2.5, lim_c * 3)
    for lo, hi, z in [(0.1, lim_a, "A"), (lim_a, lim_b, "B"), (lim_b, lim_c, "C"), (lim_c, xmax, "D")]:
        ax.axvspan(lo, hi, color=ZONE_SHADE[z], alpha=0.12, lw=0)
        ax.text((lo * hi) ** 0.5, len(m) - 0.3, f"Zone {z}", ha="center", va="bottom", fontsize=8,
                color=ZONE_SHADE[z], fontweight="bold")
    y = range(len(m))
    ax.hlines(y, 0.1, m["Max (mm/s)"], color="#90A4AE", lw=1)
    ax.scatter(m["Max (mm/s)"], y, s=70, zorder=3, c=[COLORS[c] for c in m["Condition"]], edgecolor="#263238", lw=0.6)
    for yi, (v, p) in enumerate(zip(m["Max (mm/s)"], m["Max point"])):
        ax.text(v * 1.12, yi, f"{v:g}  ({p})", va="center", fontsize=8)
    ax.set_yticks(list(y), m["Equipment"])
    ax.set_xscale("log")
    ax.set_xlim(0.2, xmax)
    ax.set_ylim(-0.7, len(m) + 0.3)
    ax.set_xlabel("Max overall velocity, mm/s RMS (log scale)")
    ax.set_title(f"{area} – every evaluated machine against ISO 20816 zones ({FOUNDATION} foundation)",
                 loc="left", fontweight="bold")
    ax.legend(handles=[Patch(color=COLORS[c], label=c) for c in ["Green", "Yellow", "Orange", "Red"]],
              title="Report condition", loc="lower right", fontsize=8)
    plt.tight_layout(); plt.show()

    # ---- Chart 2: condition donut | risk matrix | fault Pareto ----
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.3), gridspec_kw={"width_ratios": [1, 1.1, 1.5]})
    counts = a["Condition"].value_counts().reindex(list(COLORS), fill_value=0)
    counts = counts[counts > 0]
    ax[0].pie(counts, colors=[COLORS[c] for c in counts.index], startangle=90, counterclock=False,
              wedgeprops={"width": 0.38, "edgecolor": "white"},
              labels=[f"{c}\n{v}" for c, v in counts.items()], labeldistance=1.15, textprops={"fontsize": 8})
    ax[0].text(0, 0, f"{n}\nmachines", ha="center", va="center", fontsize=11, fontweight="bold")
    ax[0].set_title("Condition mix", fontweight="bold")

    mat = (ev.pivot_table(index="Criticality", columns="Condition", values="No.", aggfunc="count",
                          fill_value=0, observed=False)
             .reindex(index=["A", "B", "C"], columns=["Green", "Yellow", "Orange", "Red"], fill_value=0))
    ax[1].imshow(np.outer([3, 2, 1], [1, 2, 3, 4]), cmap="RdYlGn_r", alpha=0.75, vmin=1, vmax=12)
    for i in range(3):
        for j in range(4):
            v = int(mat.iloc[i, j])
            ax[1].text(j, i, v if v else "·", ha="center", va="center", fontsize=13, fontweight="bold")
    ax[1].set_xticks(range(4), ["Green", "Yellow", "Orange", "Red"])
    ax[1].set_yticks(range(3), ["A high", "B med", "C low"])
    ax[1].set_xlabel("Severity"); ax[1].set_ylabel("Criticality")
    ax[1].set_title("Risk matrix (machines)", fontweight="bold")
    for s in ax[1].spines.values():
        s.set_visible(False)

    if len(fm_a):
        ax[2].barh(fm_a.index[::-1], fm_a.values[::-1], color="#12476b")
        cum = (fm_a.cumsum() / fm_a.sum() * 100).round()
        for yi, (v, c) in enumerate(zip(fm_a.values[::-1], cum.values[::-1])):
            ax[2].text(v + 0.1, yi, f"{v}  ({c:.0f}% cum.)", va="center", fontsize=8)
        ax[2].set_xlim(0, fm_a.max() * 1.5)
    ax[2].set_title("Fault-mode Pareto", fontweight="bold")
    ax[2].set_xlabel("Occurrences")
    plt.tight_layout(); plt.show()

    # ---- Tables ----
    work_a = (a.sort_values(["Risk score", "Max (mm/s)"], ascending=False)
                [["Priority", "Risk score", "Equipment", "Functional Location", "Criticality", "Condition",
                  "ISO zone", "Max (mm/s)", "Max point", "Fault modes", "No healthy standby",
                  "Recommendation", "Action window"]].reset_index(drop=True))
    work_a["Fault modes"] = work_a["Fault modes"].apply(", ".join)
    work_a.index += 1

    p12 = work_a[work_a["Priority"].isin(["P1", "P2"])]
    display(HTML(f"<h4>🔧 {area} – P1/P2 work list ({len(p12)} items)</h4>"))
    if len(p12):
        display(styled(p12, "Priority", PRI_COLORS))
    else:
        print("No P1/P2 items.")

    display(HTML(f"<h4>⚠️ {area} – systems with no healthy standby</h4>"))
    if len(lost):
        display(lost.rename(columns={"Not_evaluated": "Not evaluated"}))
    else:
        print("None – every multi-unit system has at least one healthy unit.")

    survey = a[a["Condition"] == "Not Evaluated"][["Equipment", "Functional Location", "Criticality", "Priority"]]
    display(HTML(f"<h4>📋 {area} – not evaluated ({len(survey)}) – baseline needed before start-up</h4>"))
    if len(survey):
        display(survey.sort_values("Criticality").reset_index(drop=True))
    else:
        print("All machines evaluated.")
    return work_a

area_work = {}

---
## Unit 03

In [ ]:
# 5) Unit 03 full report
area_work["Unit 03"] = area_report("Unit 03")

---
## Unit 04

In [ ]:
# 6) Unit 04 full report
area_work["Unit 04"] = area_report("Unit 04")

---
## Common PH#2

In [ ]:
# 7) Common PH#2 full report
area_work["Common PH#2"] = area_report("Common PH#2")

---
## Area comparison

In [ ]:
# 8) Side-by-side comparison of the three areas
AREAS = ["Unit 03", "Unit 04", "Common PH#2"]
comp = pd.DataFrame({
    "Equipment": rel.groupby("Area").size(),
    "Evaluated %": rel.groupby("Area")["Condition"].apply(lambda s: (s != "Not Evaluated").mean() * 100),
    "Health index": rel[rel["Condition"] != "Not Evaluated"].groupby("Area")["Condition"]
                       .apply(lambda s: s.astype(str).map(HI).mean()),
    "Orange/Red": rel.groupby("Area")["Condition"].apply(lambda s: s.isin(["Orange", "Red"]).sum()),
    "P1": rel.groupby("Area")["Priority"].apply(lambda s: (s == "P1").sum()),
    "P2": rel.groupby("Area")["Priority"].apply(lambda s: (s == "P2").sum()),
    "Max reading (mm/s)": rel.groupby("Area")["Max (mm/s)"].max(),
}).reindex(AREAS)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(AREAS))
hc = ["#00B050" if h >= 75 else "#F79646" if h >= 50 else "#E53935" for h in comp["Health index"]]
ax[0].bar(x - 0.2, comp["Health index"], 0.4, color=hc, label="Health index")
ax[0].bar(x + 0.2, comp["Evaluated %"], 0.4, color="#B0BEC5", label="Evaluated %")
for xi, h, e in zip(x, comp["Health index"], comp["Evaluated %"]):
    ax[0].text(xi - 0.2, h + 2, f"{h:.0f}", ha="center", fontweight="bold")
    ax[0].text(xi + 0.2, e + 2, f"{e:.0f}%", ha="center")
ax[0].set_xticks(x, AREAS); ax[0].set_ylim(0, 115); ax[0].legend(frameon=False)
ax[0].set_title("Health index vs evaluation coverage", fontweight="bold")

stack = pd.crosstab(rel["Area"], rel["Priority"]).reindex(index=AREAS, columns=list(PRI_COLORS), fill_value=0)
left = np.zeros(len(AREAS))
for p in PRI_COLORS:
    ax[1].barh(AREAS, stack[p], left=left, color=PRI_COLORS[p], label=p, edgecolor="white")
    for i, (l, v) in enumerate(zip(left, stack[p])):
        if v:
            ax[1].text(l + v / 2, i, int(v), ha="center", va="center", fontsize=9)
    left += stack[p].values
ax[1].invert_yaxis(); ax[1].legend(ncol=5, frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.3))
ax[1].set_title("Maintenance priority by area", fontweight="bold")
plt.tight_layout(); plt.show()

display(comp.style.format({"Evaluated %": "{:.0f}%", "Health index": "{:.0f}", "Max reading (mm/s)": "{:g}"})
            .background_gradient(subset=["Health index"], cmap="RdYlGn", vmin=0, vmax=100)
            .background_gradient(subset=["Orange/Red", "P1"], cmap="Reds"))

### Search equipment

In [ ]:
# 9) Filter: change these values and re-run the cell
AREA = "All"          # "All", "Unit 03", "Unit 04", "Common PH#2"
CONDITION = "All"     # "All", "Green", "Yellow", "Orange", "Red", "Not Evaluated"
PRIORITY = "All"      # "All", "P1", "P2", "P3", "P4", "Survey"
KEYWORD = ""          # e.g. "Condensate", "Fan", "W823"

view = rel.copy()
if AREA != "All":
    view = view[view["Area"] == AREA]
if CONDITION != "All":
    view = view[view["Condition"] == CONDITION]
if PRIORITY != "All":
    view = view[view["Priority"] == PRIORITY]
if KEYWORD:
    mask = view[["Equipment", "Functional Location", "Finding"]].apply(
        lambda c: c.astype(str).str.contains(KEYWORD, case=False)).any(axis=1)
    view = view[mask]
cols = ["Area", "Equipment", "Functional Location", "Condition", "ISO zone", "Max (mm/s)", "Max point",
        "Criticality", "Priority", "Finding", "Recommendation"]
display(styled(view[cols].reset_index(drop=True), "Condition", COLORS))
print(f"{len(view)} rows")

In [ ]:
# 10) Export: one sheet per area + comparison (downloads in Colab)
OUT = "Vibration_Analysis_Before_TI.xlsx"
with pd.ExcelWriter(OUT) as xw:
    comp.to_excel(xw, sheet_name="Area comparison")
    for area in AREAS:
        area_work[area].to_excel(xw, sheet_name=area.replace("#", "")[:31])
    sys_stats.to_excel(xw, sheet_name="Redundancy")
    exp = rel.copy()
    exp["Fault modes"] = exp["Fault modes"].apply(", ".join)
    exp.to_excel(xw, sheet_name="All equipment", index=False)
print("Saved", OUT)
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass